In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from mimari import get_segmentation_model, get_loss_function

# Cihaz ayarı (GPU varsa kullan)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

# Dosya Yolların
train_img_dir = r"C:\Users\Ayberk\Desktop\FundusSegmentasyonData\Train_Image"
train_mask_dir = r"C:\Users\Ayberk\Desktop\FundusSegmentasyonData\Train_Mask"
val_img_dir = r"C:\Users\Ayberk\Desktop\FundusSegmentasyonData\Val_Image"
val_mask_dir = r"C:\Users\Ayberk\Desktop\FundusSegmentasyonData\Val_Mask"

In [ ]:
class FundusDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        # Sadece .jpg resimlerini listeye al
        self.images = [f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        # 1. Resim ismini al (Örn: "resim1.jpg")
        img_name = self.images[index]
        img_path = os.path.join(self.img_dir, img_name)
        
        # 2. Maske ismini oluştur (Örn: "resim1.jpg" -> "resim1.png")
        # splitext dosya adını ve uzantıyı ayırır: ('resim1', '.jpg')
        base_name = os.path.splitext(img_name)[0]
        mask_name = base_name + ".png"
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        # Görüntüyü oku
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Maskeyi oku
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Hata kontrolü (Dosya eksikse hangi dosya olduğunu söyler)
        if mask is None:
            raise FileNotFoundError(f"Maske bulunamadı: {mask_path}")
        
        # Binary hale getir (0.0 - 1.0)
        mask = np.float32(mask > 127) 

        if self.transform:
            augmentations = self.transform(image=image, mask=mask)
            image = augmentations["image"]
            mask = augmentations["mask"]

        return image, mask.unsqueeze(0)
        

# Veri Artırma (Augmentation) - Yetişkin verisinde çeşitliliği artırmak bebek verisine geçişi kolaylaştırır
train_transform = A.Compose([
    A.Resize(384, 384), # EfficientNet-B3 için 384x384 veya 512x512 idealdir
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(384, 384),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
train_ds = FundusDataset(train_img_dir, train_mask_dir, transform=train_transform)
val_ds = FundusDataset(val_img_dir, val_mask_dir, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0,pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

# Model, Loss ve Optimizer
model = get_segmentation_model().to(device)
criterion = get_loss_function()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # Yetişkin eğitimi için standart LR



In [ ]:
from tqdm import tqdm

num_epochs = 50
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # --- EĞİTİM AŞAMASI ---
    model.train()
    train_loss = 0
    
    # tqdm ile train_loader'ı sarıyoruz
    train_loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] - Train", leave=False)
    
    for images, masks in train_loop:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Anlık loss değerini tqdm barının sağında göster
        train_loop.set_postfix(loss=loss.item())
    
    avg_train_loss = train_loss / len(train_loader)

    # --- DOĞRULAMA (VALIDATION) AŞAMASI ---
    model.eval()
    val_loss = 0
    val_loop = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] - Val", leave=False)
    
    with torch.no_grad():
        for images, masks in val_loop:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            val_loop.set_postfix(loss=loss.item())
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Epoch özetini yazdır
    print(f"Epoch [{epoch+1}/{num_epochs}] Tamamlandı | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Model Kaydetme
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_yetiskin_model.pth")
        print(f"--> Başarı Artışı! En iyi model kaydedildi (Loss: {best_val_loss:.4f})")
    print("-" * 30)